In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

with open('/content/drive/MyDrive/NLP/Лабораторная 3/text.txt', 'r', encoding='cp1251') as f:
# with open('/content/channel_messages.txt', 'r') as f:
# with open('/content/text.txt', 'r', encoding='cp1251') as f:
    text = f.read()

text = clean_text(text)
vocab = sorted(set(text))

char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = {i: u for i, u in enumerate(vocab)}

text_as_int = [char2idx[c] for c in text]

print(f'Количество уникальных символов: {len(vocab)}')
print(f'Пример словаря символов: {list(char2idx.items())[:10]}')
print(f'Первые 100 индексов текста: {text_as_int[:100]}')

Количество уникальных символов: 79
Пример словаря символов: [(' ', 0), ('!', 1), (',', 2), ('-', 3), ('.', 4), ('0', 5), ('1', 6), ('4', 7), ('5', 8), ('8', 9)]
Первые 100 индексов текста: [32, 61, 50, 49, 53, 62, 56, 59, 47, 53, 50, 0, 20, 0, 57, 59, 56, 59, 49, 59, 62, 63, 53, 0, 76, 0, 66, 59, 61, 59, 69, 59, 0, 52, 58, 45, 56, 0, 46, 45, 61, 59, 58, 45, 0, 29, 75, 58, 66, 45, 64, 52, 50, 58, 45, 4, 0, 20, 0, 63, 59, 0, 47, 61, 50, 57, 76, 0, 50, 57, 64, 0, 51, 53, 56, 59, 62, 73, 0, 59, 68, 50, 58, 73, 0, 63, 61, 64, 49, 58, 59, 4, 0, 23, 48, 59, 0, 56, 53, 67]


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'http\S+|t\.me/\S+', '', text)

    text = re.sub(r'#\S+', '', text)

    text = re.sub(r'\[.*?\]', '', text)

    text = re.sub(r'\(.*?\)', '', text)

    text = re.sub(r'[*_@•–—]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


RNN учится предсказывать следующий символ по предыдущим.
Нужно разбить текст на "входные последовательности" и "цели" (следующий символ).

In [ ]:
import torch

seq_length = 100
examples_per_epoch = len(text_as_int) // seq_length

char_dataset = torch.tensor(text_as_int)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

sequences = char_dataset.unfold(0, seq_length + 1, step=1)

dataset = [split_input_target(seq) for seq in sequences]

In [ ]:
from torch.utils.data import DataLoader

batch_size = 64

# Функция для коллатирования батча
def collate_fn(batch):
    inputs, targets = zip(*batch)
    return torch.stack(inputs), torch.stack(targets)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Используется устройство: {device}')

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(CharRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)


Используется устройство: cpu


In [ ]:
vocab_size = len(vocab)
embedding_dim = 256
hidden_dim = 512
num_layers = 2

model = CharRNN(vocab_size, embedding_dim, hidden_dim, num_layers).to(device)

print(model)


CharRNN(
  (embedding): Embedding(79, 256)
  (rnn): RNN(256, 512, num_layers=2, batch_first=True)
  (fc): Linear(in_features=512, out_features=79, bias=True)
)


In [ ]:
import numpy as np

def get_batches(data, batch_size, seq_length):
    num_batches = int(len(data) / (batch_size * seq_length))
    data = data[:num_batches * batch_size * seq_length]

    data = np.array(data)

    # Ресайпим данные для разделения на batch_size
    data = data.reshape((batch_size, -1))

    for n in range(0, data.shape[1], seq_length):
        x = data[:, n:n+seq_length]
        y = np.zeros_like(x)
        if n + seq_length < data.shape[1]:
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, n+seq_length]
        else:
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, 0]
        yield torch.tensor(x, dtype=torch.long).to(device), torch.tensor(y, dtype=torch.long).to(device)


In [ ]:
import torch.optim as optim

def train(model, data, epochs, batch_size, seq_length, lr, clip=5):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        hidden = model.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_length):
            hidden = hidden.detach()
            optimizer.zero_grad()

            output, hidden = model(x, hidden)

            # output: (batch_size, seq_length, vocab_size)
            # нужно привести в форму (batch_size * seq_length, vocab_size)
            loss = criterion(output.view(batch_size * seq_length, -1), y.reshape(-1))
            loss.backward()

            # Градиентный клиппинг для стабильности обучения
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        if (epoch + 1) % 10 == 0:
          print(f'Epoch: {epoch + 1}/{epochs}... Loss: {loss.item()}')


In [ ]:
# Гиперпараметры обучения
batch_size = 256
seq_length = 100
epochs = 300
learning_rate = 0.00001

train(model, text_as_int, epochs, batch_size, seq_length, learning_rate)

Epoch: 10/300... Loss: 0.001024558674544096
Epoch: 20/300... Loss: 0.0010050679557025433
Epoch: 30/300... Loss: 0.0009861752623692155
Epoch: 40/300... Loss: 0.0009679296636022627
Epoch: 50/300... Loss: 0.0009502556640654802
Epoch: 60/300... Loss: 0.0009331578039564192
Epoch: 70/300... Loss: 0.0009166738600470126
Epoch: 80/300... Loss: 0.0009007100597955287
Epoch: 90/300... Loss: 0.0008848035940900445
Epoch: 100/300... Loss: 0.0008685729699209332
Epoch: 110/300... Loss: 0.0008521848358213902
Epoch: 120/300... Loss: 0.0008359260973520577
Epoch: 130/300... Loss: 0.0008199741132557392
Epoch: 140/300... Loss: 0.0008044270216487348
Epoch: 150/300... Loss: 0.0007892940193414688
Epoch: 160/300... Loss: 0.0007745225448161364
Epoch: 170/300... Loss: 0.0007606760482303798
Epoch: 180/300... Loss: 0.0007470612763427198
Epoch: 190/300... Loss: 0.0007338201394304633
Epoch: 200/300... Loss: 0.000720949552487582
Epoch: 210/300... Loss: 0.000708242179825902
Epoch: 220/300... Loss: 0.0006953771226108074


In [ ]:
def generate_text(model, start_string, gen_size=1000, temperature=1.0):
    model.eval()
    input_eval = torch.tensor([char2idx[s] for s in start_string], device=device).unsqueeze(0)
    hidden = model.init_hidden(1)

    generated_text = start_string

    with torch.no_grad():
        for _ in range(gen_size):
            output, hidden = model(input_eval, hidden)

            output = output[:, -1, :] / temperature
            probabilities = torch.softmax(output, dim=-1).squeeze()

            next_char_idx = torch.multinomial(probabilities, 1).item()
            next_char = idx2char[next_char_idx]

            generated_text += next_char

            input_eval = torch.tensor([[next_char_idx]], device=device)

    return generated_text

In [ ]:
start_string = "Собака бежала "
generated = generate_text(model, start_string, gen_size=1000, temperature=0.8)
generated

'Собака бежала не было видно, торчали одни уши. Тут нужна была немедленная и самая решительная помощь. Я крепко сжал бока лошади своими ногами, схватился рукой за свой собственный чуб и… представьте, вытащил себя вместе с конём из этого топкого болота. О, да, у меня тогда была силушка не та, совсем не та, что теперь! На другой день я опять поехал на любимой лошади по своему обширному имению. Дел было много, и вернуться пришлём. Они подняли меня на смех. Меня разозлило это, и я заявил ему, что чутью своей Траи я доверяю больше, чем глазам всех моряков нашего экипажа, и предложил ему пари на  место, сельате жилой о кругосити куропа от лууловым, камие. Она успели мне всегда рассказами о своих приключениях. Теперь я объясняю свою страсть к путешествиям не только врождённой склонностью, но и тем, что я следовал примеру отца. Я наслушался так много всяких занимательных рассказов о приключениями на моих глазах потянул верёвку и выдернул с корнем все деревья, как будто это был маленький кустик

In [ ]:
start_string = "Собака бежала "
generated = generate_text(model, start_string, gen_size=1000, temperature=0.8)
generated

NameError: name 'generate_text' is not defined

In [ ]:
# Сохраняем веса модели
torch.save(model.state_dict(), '/content/drive/MyDrive/NLP/Лабораторная 3/RNN/char_rnn_model.pth')

# Сохраняем словари (нужно для декодирования при генерации)
import pickle

with open('char2idx.pkl', 'wb') as f:
    pickle.dump(char2idx, f)

with open('idx2char.pkl', 'wb') as f:
    pickle.dump(idx2char, f)


In [ ]:
import pickle
# Загружаем словари
with open('/content/drive/MyDrive/NLP/Лабораторная 3/RNN/char2idx_Жириновский.pkl', 'rb') as f:
    char2idx = pickle.load(f)

with open('/content/drive/MyDrive/NLP/Лабораторная 3/RNN/idx2char_Жириновский.pkl', 'rb') as f:
    idx2char = pickle.load(f)

# Обновляем размер словаря
vocab_size = len(char2idx)

# Создаем модель и загружаем веса
embed_size = 256
hidden_size = 512
num_layers = 2

model_new = CharRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
model_new.load_state_dict(torch.load('/content/drive/MyDrive/NLP/Лабораторная 3/RNN/char_rnn_model_Жириновский.pth'))
model_new.eval()


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
import pickle
import torch

with open('/content/drive/MyDrive/NLP/Лабораторная 3/RNN/char2idx_Жириновский.pkl', 'rb') as f:
    char2idx = pickle.load(f)

with open('/content/drive/MyDrive/NLP/Лабораторная 3/RNN/idx2char_Жириновский.pkl', 'rb') as f:
    idx2char = pickle.load(f)

vocab_size = len(char2idx)

embed_size = 256
hidden_size = 512
num_layers = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_new = CharRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)

model_new.load_state_dict(
    torch.load(
        '/content/drive/MyDrive/NLP/Лабораторная 3/RNN/char_rnn_model_Жириновский.pth',
        map_location=device
    )
)
model_new.eval()

CharRNN(
  (embedding): Embedding(137, 256)
  (rnn): RNN(256, 512, num_layers=2, batch_first=True)
  (fc): Linear(in_features=512, out_features=137, bias=True)
)

In [ ]:
start_string = "В итоге для нашей экономики ноль по всем направлениям."
generated = generate_text(model_new, start_string, gen_size=1000, temperature=0.8)
generated

'В итоге для нашей экономики ноль по всем направлениям. По сути, мировым правительством ставится ультиматум: не хотите мировой войны платите. И мы платим им дань. Продолжаем платить  стали понимать, что есть городская культура, что существуют унитазы, кушать надо вилкой, следует мыть руки и желательно - всё тело. А чно докопатают, международное движение, следствием которого будет вечный мир и восторжествуют идеалы свободы, равенсвит так, все это и претва говорил Ж. ( В соцеании, а они побадии, Русской почти в нах это традиция, которая имеет свое название бакшиш. Там это нормально, начальнику всегда все несут подарки. Но наши люди растут там, все это видят и этим заражаются. Потом они приезж, азвеличии сонтируяст ммернур обыовы. Вот тем во внни- мирового поласеливьюе, отсучен свое, потому что конерная детами, кужде опостали об это только посвятенным ханом одного их кочто в каждую место на говор. А Какажитиль. Продолжил о соедние в отправь: «Счерьяд в алларании, акрбывайи вы становились 